In [ ]:
import numpy as np
import pandas as pd
import random as rn
import os
import cv2
import tensorflow.random as tfr
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import matplotlib.pyplot as plt
import seaborn as sns
from skimage import color

seed = 8
np.random.seed(seed)
rn.seed(seed)
tfr.set_seed(seed)

print("Запуск...")

data_path = 'C:/Users/User/Desktop/pneu/input/'
train_path = data_path + 'train/'
test_path = data_path + 'test/'
val_path = data_path + 'val/'

img_size = 196

def arr_img(dt_pths):
    lbls = ['PNEUMONIA', 'NORMAL']
    imgs = []

    for dt_pth in dt_pths:
        for lbl in lbls:
            crr_pth = os.path.join(dt_pth, lbl)

            for img_name in os.listdir(crr_pth):
                img_pth = os.path.join(crr_pth, img_name)
                img = cv2.imread(img_pth)

                if img is not None:
                    img = cv2.resize(img, (img_size, img_size))
                    imgs.append([img, lbl])

    return np.array(imgs, dtype=object)

train = arr_img([train_path])
test = arr_img([val_path, test_path])

np.random.shuffle(train)
np.random.shuffle(test)

X_train, Y_train = [], []
X_test, Y_test = [], []

for val, lbl in train:
    X_train.append(val)
    Y_train.append(0 if lbl == 'NORMAL' else 1)

for val, lbl in test:
    X_test.append(val)
    Y_test.append(0 if lbl == 'NORMAL' else 1)

X_train = np.array(X_train)
X_test = np.array(X_test)

X_train = color.rgb2gray(X_train)
X_test = color.rgb2gray(X_test)

X_train = X_train.reshape(-1, img_size, img_size, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, img_size, img_size, 1).astype('float32') / 255.0

y_train = to_categorical(Y_train, 2)
y_test = to_categorical(Y_test, 2)

dtg = ImageDataGenerator(
    width_shift_range=0.1,
    height_shift_range=0.1,
    rotation_range=15,
    zoom_range=0.1
)

dtg.fit(X_train)
train_gen = dtg.flow(X_train, y_train, batch_size=32)

def get_md():
    return Sequential([
        Conv2D(16, (7, 7), activation='relu', padding='same', input_shape=(img_size, img_size, 1)),
        Conv2D(16, (7, 7), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((3, 3)),
        Dropout(0.2),

        Conv2D(32, (5, 5), activation='relu', padding='same'),
        Conv2D(32, (5, 5), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((3, 3)),
        Dropout(0.2),

        Conv2D(64, (3, 3), activation='relu', padding='same'),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.2),

        Conv2D(128, (3, 3), activation='relu', padding='same'),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.2),

        Conv2D(256, (3, 3), activation='relu', padding='same'),
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.2),

        Flatten(),

        Dense(1024, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),

        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),

        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(2, activation="softmax")  # 2 класса
    ])

model = get_md()

model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

model.summary()

os.makedirs('C:/Users/User/Desktop/working/', exist_ok=True)

clbk = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', patience=4),
    ModelCheckpoint('C:/Users/User/Desktop/working/pneumokort.hdf5',
                    monitor='val_loss', save_best_only=True)
]

history = model.fit(
    train_gen,
    epochs=100,
    steps_per_epoch=X_train.shape[0] // 32,
    validation_data=(X_test, y_test),
    callbacks=clbk
)

model = load_model('C:/Users/User/Desktop/working/pneumokort.hdf5')

score = model.evaluate(X_test, y_test, verbose=0)

print(f"Потери (loss): {score[0]*100:.2f}%")
print(f"Точность (accuracy): {score[1]*100:.2f}%")
print(f"Ошибка: {100 - score[1]*100:.2f}%")

def drw_ed_grf(history, keys=['accuracy', 'loss']):
    plt.figure(figsize=(20, 8))

    for i, key in enumerate(keys):
        plt.subplot(1, 2, i + 1)

        sns.lineplot(x=range(len(history.history[key])), y=history.history[key])
        sns.lineplot(x=range(len(history.history['val_' + key])), y=history.history['val_' + key])

        plt.title('Кривая обучения')
        plt.ylabel(key)
        plt.xlabel('Эпоха')
        plt.legend(['train', 'val'])

    plt.show()

drw_ed_grf(history)